# Modelado

El objetivo de este notebook es entrenar distintos modelos de regresión supervisada para
predecir los costes médicos individuales (`charges`), comparar su rendimiento y seleccionar
el modelo final mediante técnicas adecuadas de validación y ajuste de hiperparámetros.

El conjunto de test se mantiene completamente aislado hasta la fase de evaluación final.

### Importar las librerías

In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Cargar los datos

In [31]:
df = pd.read_csv("../data/insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### Feature engineering

In [32]:
df = df.drop_duplicates()

In [33]:
bins = [18, 25, 35, 45, 55, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56+"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

interval_means = {
    "18-25": 21.5,
    "26-35": 30.5,
    "36-45": 40.5,
    "46-55": 50.5,
    "56+": 66.0
}

df["age_group_mean"] = df["age_group"].map(interval_means)

### Target y variables

In [34]:
TARGET = "charges"

X = df.drop(columns=["charges", "age", "age_group"])
y = df["charges"]

### Separación TRAIN / DEV / TEST

In [35]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

### Preprocesado

In [36]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ]
)

### Función de evaluación

In [37]:
def evaluate(y_true, y_pred, model_name="model"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{model_name} -> MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.3f}")
    
    return {
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }


### Selección de algoritmos

Para abordar el problema de predicción de los costes médicos (`charges`), se han seleccionado distintos
algoritmos de regresión con el objetivo de comparar enfoques de complejidad creciente y evaluar su
capacidad para capturar las relaciones presentes en los datos.

#### Regresión lineal

In [38]:
from sklearn.linear_model import LinearRegression

lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr.fit(X_train, y_train)
evaluate(y_dev, lr.predict(X_dev), "Linear Regression (dev)")

Linear Regression (dev) -> MAE: 5548.62 | RMSE: 7306.65 | R2: 0.688


{'model': 'Linear Regression (dev)',
 'mae': 5548.616624686506,
 'rmse': np.float64(7306.6491777156025),
 'r2': 0.6879493928069116}

Se ha utilizado un modelo de regresión lineal como modelo base, ya que
proporciona un punto de referencia sencillo, interpretable y fácil de entrenar. Este tipo de modelos
permite evaluar rápidamente si existen relaciones aproximadamente lineales entre las variables
explicativas y la variable objetivo.

#### Regresión ridge

In [39]:
from sklearn.linear_model import Ridge

ridge = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

ridge.fit(X_train, y_train)
evaluate(y_dev, ridge.predict(X_dev), "Ridge (dev)")

Ridge (dev) -> MAE: 5557.52 | RMSE: 7318.39 | R2: 0.687


{'model': 'Ridge (dev)',
 'mae': 5557.524867111067,
 'rmse': np.float64(7318.391575397532),
 'r2': 0.6869456039035269}

Como extensión del modelo lineal, se ha incluido la regresión Ridge, que incorpora regularización L2
para reducir la varianza del modelo y mitigar posibles problemas de multicolinealidad, manteniendo
una buena capacidad interpretativa.

#### Regresión Random Forest

In [40]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

rf.fit(X_train, y_train)
evaluate(y_dev, rf.predict(X_dev), "Random Forest (dev)")

Random Forest (dev) -> MAE: 5225.59 | RMSE: 6746.08 | R2: 0.734


{'model': 'Random Forest (dev)',
 'mae': 5225.586048697082,
 'rmse': np.float64(6746.084613542549),
 'r2': 0.7339935979981083}

Se han considerado modelos basados en árboles, como la regresión Random Forest, lo cual
permite capturar relaciones no lineales y posibles interacciones entre variables sin requerir una
especificación explícita de las mismas. Además, estos modelos son robustos frente a outliers y a
transformaciones monotónicas de las variables.

#### Regresión Gradient Boosting

In [41]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

gbr.fit(X_train, y_train)
evaluate(y_dev, gbr.predict(X_dev), "Gradient Boosting (dev)")

Gradient Boosting (dev) -> MAE: 4720.02 | RMSE: 6223.89 | R2: 0.774


{'model': 'Gradient Boosting (dev)',
 'mae': 4720.019940686188,
 'rmse': np.float64(6223.889196653469),
 'r2': 0.7735813349460166}

Se ha incluido la regresión Gradient Boosting, ya que es un modelo de ensamblado secuencial que combina
múltiples árboles débiles para minimizar el error de forma iterativa. Este algoritmo suele ofrecer un
mejor rendimiento predictivo en problemas de regresión tabular, debido a que captura patrones complejos y
no lineales, lo que le convierte en un candidato natural para la selección final del modelo.

### Comparación de resultados

In [42]:
results = [
    evaluate(y_dev, lr.predict(X_dev), "Linear"),
    evaluate(y_dev, ridge.predict(X_dev), "Ridge"),
    evaluate(y_dev, rf.predict(X_dev), "RandomForest"),
    evaluate(y_dev, gbr.predict(X_dev), "GradientBoosting"),
]

pd.DataFrame(results).sort_values("rmse")

Linear -> MAE: 5548.62 | RMSE: 7306.65 | R2: 0.688
Ridge -> MAE: 5557.52 | RMSE: 7318.39 | R2: 0.687
RandomForest -> MAE: 5225.59 | RMSE: 6746.08 | R2: 0.734
GradientBoosting -> MAE: 4720.02 | RMSE: 6223.89 | R2: 0.774


,model,mae,rmse,r2
3,GradientBoosting,4720.019941,6223.889197,0.773581
2,RandomForest,5225.586049,6746.084614,0.733994
0,Linear,5548.616625,7306.649178,0.687949
1,Ridge,5557.524867,7318.391575,0.686946


### Selección de hiperparámetros (GridSearch)

In [43]:
X_tune = pd.concat([X_train, X_dev])
y_tune = pd.concat([y_train, y_dev])

In [44]:
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2],
}

rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

grid = GridSearchCV(
    rf_pipe,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_tune, y_tune)

grid.best_params_, -grid.best_score_

({'model__max_depth': 8,
  'model__min_samples_leaf': 2,
  'model__n_estimators': 200},
 np.float64(6930.714248347358))

### Modelo seleccionado

El modelo seleccionado es la regresión Gradient Boosting, ya que presenta menores errores (RMSE y MAE) de todos los modelos evaluados que hay en el conjunto de validación, además de tener una mayor capacidad para capturar relaciones no lineales entre las variables.
